# Inspection Demo · Notebook 启动器

按顺序执行。Notebook 内核支持 Python 3.10+；应用使用 `uv` 建立独立的 Python 3.12 `.venv`，并安装仓库锁定的依赖。默认 CPU，可选 CUDA。首次安装与私有资产下载需要网络。

本地 Jupyter：启动后打开本机 UI。Colab：仅通过本机 HTTP 提交查询并在 Notebook 查看结果，不提供 Web UI、公开端口或隧道。查询使用你的 OpenAI API key，会产生 API 费用。

如果 Windows Application Control 阻止 Python，请使用管理员批准的 Python/运行环境后重试；不要改名、复制可执行文件或关闭策略。完整步骤见 `docs/NOTEBOOK_QUICKSTART.md`。


## 1. 获取仓库（已有完整 checkout 可跳过）

推荐先在终端执行 `git clone https://github.com/ShellWee/inspection-demo.git`，再打开本文件。更新已有仓库时，在仓库目录执行 `git pull --ff-only`。

如果只在 Colab 打开了 `.ipynb`，可在下一格填写仓库 HTTPS URL 来克隆。私有仓库访问必须事先配置 Git 凭据；不要把 token 写进 URL、单元格或输出。留空会跳过克隆。


In [ ]:
from pathlib import Path
import os
import subprocess
from urllib.parse import urlsplit

GIT_URL = ""  # Set to https://github.com/ShellWee/inspection-demo.git only if no checkout exists. Never include a token.
CHECKOUT_DIR = Path.cwd() / "inspection-demo-hf"

if GIT_URL:
    parsed_url = urlsplit(GIT_URL)
    if (parsed_url.scheme != "https" or not parsed_url.hostname
            or parsed_url.username or parsed_url.password
            or parsed_url.query or parsed_url.fragment):
        raise ValueError("Use a repository HTTPS URL without credentials.")
    if CHECKOUT_DIR.exists():
        print("Checkout directory exists; update it with git pull --ff-only if needed.")
    else:
        git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        subprocess.run(["git", "clone", GIT_URL, str(CHECKOUT_DIR)],
                       check=True, env=git_env)
else:
    print("Using the existing checkout.")


## 2. 配置

`ASSETS_DIR` 默认是仓库内的 `assets`；也可改成已有资产包的路径。缺少资产时，将 `DOWNLOAD_ASSETS` 设为 `True`，使用现有 Hugging Face 登录下载固定版本的私有数据集。私有资产权限与 Git 仓库权限相互独立。

`DEVICE="cuda"` 仅适用于有兼容 NVIDIA GPU 的环境；选择 CUDA 后启动器会验证设备。Notebook 内核不需要安装应用的 Torch 或 FastAPI。


In [ ]:
from pathlib import Path
import os
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Use an approved Python 3.10+ notebook kernel.")

def find_repo(start):
    start = Path(start).resolve()
    candidates = [start, *start.parents,
                  start / "inspection-demo-hf", start / "inspection-demo"]
    for candidate in candidates:
        if ((candidate / "scripts" / "demo_launcher.py").is_file()
                and (candidate / "pyproject.toml").is_file()):
            return candidate
    raise RuntimeError("Clone the complete repository, then open this notebook from that checkout.")

REPO_ROOT = find_repo(Path.cwd())
ASSETS_DIR = REPO_ROOT / "assets"  # Or Path("/path/to/existing/asset-bundle")
STATE_DIR = REPO_ROOT / ".demo-state"
DEVICE = "cpu"                   # "cpu" or "cuda"
DOWNLOAD_ASSETS = False           # Explicit opt-in to private HF download
PORT = 7860
MODEL_ID = "gpt-4.1"              # Also: "gpt-5", "gpt-5.6-luna"
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
BASE_URL = f"http://127.0.0.1:{PORT}"

print("Repository:", REPO_ROOT)
print("Assets:", ASSETS_DIR)
print("Mode:", "Colab HTTP results" if IN_COLAB else "Local browser + HTTP results")


## 3. 安装应用环境

先确保当前内核能调用 `uv`。需要由 Notebook 安装时，将下一格的 `INSTALL_UV` 改成 `True`；它只安装固定版本的 `uv` 到当前内核环境。随后 setup 使用 `uv.lock` 建立应用的 Python 3.12 环境。仓库已包含构建好的前端，启动不需要 Node.js。


In [ ]:
import subprocess
import sys

INSTALL_UV = False
if INSTALL_UV:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.9"], check=True)


In [ ]:
import subprocess
import sys

def run_launcher(*arguments):
    return subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "demo_launcher.py"), *map(str, arguments)],
        cwd=REPO_ROOT, check=True,
    )

run_launcher("setup", "--device", DEVICE)


## 4. 验证或下载资产

已有完整资产包时保持 `DOWNLOAD_ASSETS=False`。需要下载时，先在当前运行环境完成 `hf auth login`（只读权限并获准访问私有数据集），再设为 `True`。下载使用固定数据集版本；本格不接收 token。


In [ ]:
asset_arguments = ["assets", "--assets-dir", ASSETS_DIR]
if DOWNLOAD_ASSETS:
    asset_arguments.append("--download")
run_launcher(*asset_arguments)


## 5. 启动与检查

服务只监听 `127.0.0.1`。启动器等待服务就绪，并把进程信息保存在 `STATE_DIR`，因此关闭浏览器单元格不等于停止服务；最后一格可显式停止。重启 Notebook 内核后，重新运行配置和函数定义即可用同一个 state 目录检查/停止服务。


In [ ]:
run_launcher("start", "--assets-dir", ASSETS_DIR, "--device", DEVICE,
             "--port", PORT, "--state-dir", STATE_DIR)
if IN_COLAB:
    print("Ready. Use the query cell below; no public browser URL is created.")
else:
    print(f"Open on this computer: {BASE_URL}")


In [ ]:
run_launcher("status", "--state-dir", STATE_DIR)


## 6. 可选：在 Notebook 中提交查询

执行付费查询前，将 `RUN_QUERY` 改成 `True`。输入英文原始问题，再通过隐藏输入提供 OpenAI API key。客户端只提交一次，随后轮询到 `completed`、`abstained`、`failed` 或 `cancelled`。

密钥只用于请求，不写进环境变量、命令行或文件；响应中的密钥会被遮盖，结束后释放 Notebook 的密钥变量。超时或中断不会取消已提交的任务，避免立即重试造成重复调用。结果包含你的查询与建筑信息，分享/提交 Notebook 前请清除所有输出。


In [ ]:
import getpass
import importlib.util
import json
import warnings

RUN_QUERY = False
if RUN_QUERY:
    spec = importlib.util.spec_from_file_location(
        "inspection_notebook_client", REPO_ROOT / "scripts" / "notebook_client.py")
    notebook_client = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(notebook_client)
    query = input("English inspection query (3–2000 characters): ")
    api_key = None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", getpass.GetPassWarning)
            api_key = getpass.getpass("OpenAI API key (hidden): ")
        run_result = notebook_client.run_grounding(
            base_url=BASE_URL, query=query, model_id=MODEL_ID, api_key=api_key)
        print(json.dumps(run_result, ensure_ascii=False, indent=2))
    except notebook_client.NotebookClientError as error:
        print(str(error))
    except getpass.GetPassWarning:
        print("Hidden input is unavailable. Use a notebook frontend that supports getpass.")
    except KeyboardInterrupt:
        print("Client interrupted; an accepted backend run may still be running.")
    except Exception as error:
        print(f"Query stopped ({type(error).__name__}); no diagnostic body is displayed.")
    finally:
        api_key = ""
        del api_key
else:
    print("No API call made. Set RUN_QUERY=True when ready.")


## 7. 停止服务

需要停止时将 `STOP_SERVER` 设为 `True` 并执行。本格默认跳过，避免 “Run All” 刚启动就关闭服务。状态目录中的本地运行数据由启动器管理；停止后可再运行启动格。


In [ ]:
STOP_SERVER = False
if STOP_SERVER:
    run_launcher("stop", "--state-dir", STATE_DIR)
else:
    print("Server left running. Set STOP_SERVER=True to stop it.")
